# CS 263 Assignment 3: Knowledge Retrieval & RAG

## Deadline: 3/11/2026 at 11:59pm


## Outline
- Section 0: Installation & Setup (10 points)
- Section 1: Context-Free Generation (15 points)
- Section 2: Build Dense Retriever (20 points)
- Section 3: Retrieval-Augmented Generation (15 points)

## Instructions
- Follow the instructions and fill in the code for the sections marked with `# TODO`.
- **DO NOT** modify the checking/grading cells. Modifying these cells is strictly prohibited and will be treated as an academic integrity violation which will result in 0 score for this assignment and escalation to The Office of Student Conduct at UCLA.

## Submission
- **Execution**: Ensure all cells have been run, and outputs are displayed before submission.
- **File Naming**: Save your completed notebook with outputs as `hw3.ipynb`.
- **Upload**: Submit your `hw3.ipynb` file to Gradescope.

Failure to follow these instructions will result in the autograder failing, which will automatically result in 0 points. **No regrading will be done** for submissions with incorrect file names or formats.

## Section 0: Installation & Setup

Tips:
- You can use a GPU to speed up the process. Select "Runtime" > "Change runtime type" > "GPU" in the Colab settings.
- The model we are going to use is [Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct). Please visit the model page on Hugging Face and accept the terms of the license. Typically, access is granted within 24 hours.
- Go to huggingface -> Profile icon on the upper-right corner -> "Settings" -> "Access Tokens" to get the key for logging in

In [1]:
!pip install -q transformers datasets sentence-transformers evaluate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [3]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### Load Model & Dataset

We will use the pre-trained model: `meta-llama/Llama-3.2-1B-Instruct`

The dataset we use is `rag-datasets/rag-mini-bioasq`. This dataset contains two relevant splits
- `question-answer-passages`: contains biomedical questions, human-annotated reference answers, and ground-truth supporting passage IDs.
- `text-corpus`: a collection of biomedical passages, serving as the retrieval database.

In [8]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto" if device == "cuda" else None
)

model.eval()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [11]:
qa_data = load_dataset("rag-datasets/rag-mini-bioasq", "question-answer-passages")["test"]
corpus_data = load_dataset("rag-datasets/rag-mini-bioasq", "text-corpus")['passages']

print("QA subset size:", len(qa_data))
print("Corpus size:", len(corpus_data))

QA subset size: 4719
Corpus size: 40221


### Helper Function: Text Generation (10 points)

In [16]:
def generate_answer(prompt, model, tokenizer, max_new_tokens=200):
    """
    TODO:
    - Given a prompt string, generate an answer using the Llama-3.2-1B-Instruct.
    - Steps to implement:
        1. Wrap the prompt into a chat message format:
            messages = [{"role": "user", "content": prompt}]
        2. Tokenize the messages using `tokenizer.apply_chat_template`
        3. Generate output tokens from the model using `model.generate`
            - Use `do_sample=False` for greedy generation
        4. Decode the generated tokens to a string

    Args:
        prompt (str): The input prompt or question.
        model: HuggingFace LLaMA model ("meta-llama/Llama-3-2-1B-Instruct").
        tokenizer: Corresponding tokenizer for the model.
        max_new_tokens (int): Maximum number of tokens to generate.

    Returns:
        str: The model-generated answer.
    """
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
      messages,
      return_tensors="pt",
      add_generation_prompt=True
    ).to(model.device)

    if hasattr(input_ids, 'input_ids'):
        input_ids = input_ids.input_ids

    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False)
    generated_tokens = output_ids[0][input_ids.shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer

In [17]:
# =====================================================
# DO NOT MODIFY
# =====================================================
prompt = "When was UCLA founded?"
grading_output = generate_answer(prompt=prompt, model=model, tokenizer=tokenizer, max_new_tokens=200)
expected_output = "The University of California, Los Angeles (UCLA) was founded in 1919."
print(f"Your output is: {grading_output}\nThe expected output is: {expected_output}")
if grading_output != expected_output:
    raise ValueError(f"FAILED: Incorrect response! \n\n{grading_output}\n\n{expected_output}")
print("Part 0 passed")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Your output is: The University of California, Los Angeles (UCLA) was founded in 1919.
The expected output is: The University of California, Los Angeles (UCLA) was founded in 1919.
Part 0 passed


## Section 1: Context-Free Generation (15 points)

In this section, you will evaluate a pretrained large language model on a biomedical question-answering task **without retrieval augmentation**. The goal is to measure baseline generation performance before introducing a retriever.

### Task:
Use the first 10 examples from the `rag-mini-bioasq` dataset (`question-answer-passages` split)
- Construct the following instruction prompt:
> You are a helpful chatbot.
> Answer the question as truthfully as possible.
>
> Question: [question]
> Answer:

- Generate answers using `Llama-3.2-1B-Instruct` with context-free instruction prompt.
- Compute BLEU score against ground-truth answers.

In [22]:
def build_context_free_prompt(question):
  # TODO:
    # - Write the full context-free instruction prompt as a formatted string.
    # - Insert the variable `question`.
    # - Make sure the prompt ends with "Answer:"

  prompt = (
        "You are a helpful chatbot.Answer the question as truthfully as possible.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
  return prompt

In [23]:
predictions = []
references = []

for i in range(10):
    example = qa_data[i]
    question = example["question"]
    reference_answer = example["answer"]

    # TODO: 1. Build prompt

    # TODO: 2. Generate model answer

    # TODO: 3. Store prediction and reference
    prompt = build_context_free_prompt(question)
    answer = generate_answer(prompt, model, tokenizer)
    predictions.append(answer)
    references.append(reference_answer)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end gene

In [24]:
def compute_bleu(predictions, references):
    """
    predictions: list of strings
    references: list of list of reference strings
    return average BLEU score
    """
    # TODO

    from evaluate import load
    bleu = load("bleu")
    result = bleu.compute(predictions=predictions, references=references)
    return result["bleu"]

student_bleu = compute_bleu(predictions, references)
print("Average BLEU (no retrieval):", student_bleu)

Average BLEU (no retrieval): 0.021500568233136334


In [25]:
# =====================================================
# DO NOT MODIFY
# =====================================================

assert len(predictions) == 10, "You must generate exactly 10 predictions."
assert len(references) == 10, "References length mismatch."

# Check outputs are non-empty
for p in predictions:
    if not isinstance(p, str) or len(p.strip()) == 0:
        raise ValueError("FAILED: Some predictions are empty.")
print("Part 1.1 passed")

# Compute BLEU safely
from evaluate import load
bleu = load("bleu")
bleu_score = bleu.compute(predictions=predictions, references=references)
bleu_value = bleu_score["bleu"]
print(f"Computed BLEU: {bleu_value}")
if abs(student_bleu - bleu_value) > 0.01:
    raise ValueError(
        "FAILED: Your BLEU implementation differs too much from official implementation."
    )
print("Part 1.2 passed")

# Set baseline expectation
expected_bleu = 0.02   # realistic baseline for biomedical zero-shot

if bleu_value < expected_bleu / 2:
    raise ValueError(
        f"FAILED: BLEU too low ({bleu_value}).\n"
        "Model may not be generating properly or prompt is incorrect."
    )

if bleu_value < expected_bleu:
    raise ValueError(
        f"FAILED: BLEU below expected threshold {expected_bleu}.\n"
        "Try improving the prompt formatting."
    )

print("Part 1.3 passed")

Part 1.1 passed
Computed BLEU: 0.021500568233136334
Part 1.2 passed
Part 1.3 passed


## Section 2: Build Dense Retriever (20 points)

In this section, you will implement a dense retrieval system using vector embeddings and cosine similarity. The goal is to retrieve relevant biomedical passages from a corpus to support question answering.

### Task Overview
You will:
- Encode the entire `text-corpus` split into dense embeddings.
- Encode each question into an embedding.
- Compute cosine similarity between question and corpus embeddings.
- Retrieve the top-k most relevant passages (k=5).
- Evaluate retrieval performance using:
  - Recall@k
  - Precision@k
  - F1@k

### Encode Corpus

#### Load Embedding Model
Load a biomedical sentence embedding model from Hugging Face:
- Use: `pritamdeka/S-PubMedBert-MS-MARCO`
- You can find the model card [here](https://huggingface.co/pritamdeka/S-PubMedBert-MS-MARCO)

Encode all passages in the corpus:
- `corpus_data` is a list of dicts with keys "passage" and "id"
  - Store passage texts in `corpus_texts`
  - Store corresponding IDs in `corpus_ids`
- Compute embeddings for all passages and store in `corpus_embeddings`
- Use `show_progress_bar=True` so you can see progress (this may take a few minutes).

In [26]:
embedder = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
corpus_texts = [doc["passage"] for doc in corpus_data]
corpus_ids = [doc["id"] for doc in corpus_data]

corpus_embeddings = embedder.encode(
    corpus_texts,
    convert_to_tensor=False,
    show_progress_bar=True
)

Batches:   0%|          | 0/1257 [00:00<?, ?it/s]

### Retrieve Top-k Passages

In [28]:
def cosine_similarity_batch(query_matrix, corpus_matrix):
    """
    TODO: Implement cosine similarity

    query_matrix: shape (M, d)
    corpus_matrix: shape (N, d)
    returns: similarity matrix (M, N)

    Compute the cosine similarity between the query and corpus embeddings.

    Requirements and tips:
    - Normalize embeddings before computing dot product.
    - Use only NumPy operations.
    - Avoid division by zero (add a small epsilon like 1e-10 when normalizing).
    """

    query_norm = query_matrix / (np.linalg.norm(query_matrix, axis=1, keepdims=True) + 1e-10)
    corpus_norm = corpus_matrix / (np.linalg.norm(corpus_matrix, axis=1, keepdims=True) + 1e-10)
    return query_norm @ corpus_norm.T

In [29]:
# ==================================================
# DO NOT MODIFY
# ==================================================

from sklearn.metrics.pairwise import cosine_similarity

query = np.random.rand(3, 768)
corpus = np.random.rand(7, 768)

try:
    manual_matrix = cosine_similarity_batch(query, corpus)
except Exception as e:
    raise ValueError(f"FAILED: Your function raised an error: {e}")

if manual_matrix.shape != (query.shape[0], corpus.shape[0]):
    raise ValueError(f"FAILED: Output shape should be {(query.shape[0], corpus.shape[0])}, got {manual_matrix.shape}")

sklearn_matrix = cosine_similarity(query, corpus)
max_diff = np.max(np.abs(manual_matrix - sklearn_matrix))
print("Max diff from sklearn:", max_diff)

if max_diff > 1e-5:
    raise ValueError("FAILED: Your cosine similarity values differ significantly from sklearn.cosine_similarity")

print("Part 2.1 passed")

Max diff from sklearn: 9.553469126899472e-12
Part 2.1 passed


In [30]:
def retrieve_top_k(question, corpus_embeddings, k=5):
    """
    Retrieve top-k passages from corpus based on cosine similarity to the question.

    Args:
        question (str): Input question string.
        corpus_embeddings (list or np.array): Encoded embeddings of corpus passages.
        corpus_ids (list): IDs corresponding to corpus passages.
        k (int): Number of top passages to retrieve.

    Returns:
        tuple:
            - top_k_indices (list of int): Indices of the top-k most similar passages in the corpus.
    """

    # TODO: Encode question
    query_embedding = embedder.encode([question], convert_to_tensor=False)
    # TODO: Compute similarities
    similarities = cosine_similarity_batch(query_embedding, np.array(corpus_embeddings))
    # TODO: Get top-k indices
    top_k_indices = np.argsort(similarities[0])[::-1][:k].tolist()
    return top_k_indices


In [31]:
# ==================================================
# DO NOT MODIFY
# ==================================================

# Test the student implementation
test_question = "What is insulin resistance?"
k_test = 5

try:
    top_k_indices = retrieve_top_k(test_question, corpus_embeddings, k=k_test)
except Exception as e:
    raise ValueError(f"FAILED: Your function raised an error: {e}")

# Check type
if not isinstance(top_k_indices, (list, np.ndarray)):
    raise ValueError("FAILED: Output should be a list or numpy array of passage IDs.")

# Check length
if len(top_k_indices) != k_test:
    raise ValueError(f"FAILED: Output length should be {k_test}, got {len(retrieved_ids)}")

try:
    retrieved_ids = [corpus_ids[idx] for idx in top_k_indices]
except Exception as e:
    raise ValueError(f"FAILED: Error mapping indices to corpus_ids: {e}")

print("Part 2.2 passed")

Part 2.2 passed


In [32]:
# ==================================================
# DO NOT MODIFY
# ==================================================

def compute_ir_metrics(retrieved_ids, ground_truth_ids):
    retrieved_set = set(retrieved_ids)
    ground_truth_set = set(ground_truth_ids)

    correct = len(retrieved_set & ground_truth_set)

    recall = correct / len(ground_truth_set) if ground_truth_set else 0
    precision = correct / len(retrieved_set) if retrieved_set else 0

    if recall + precision == 0:
        f1 = 0
    else:
        f1 = 2 * recall * precision / (recall + precision)

    return recall, precision, f1

k = 5
recalls, precisions, f1s = [], [], []

import ast
for i in range(10):
    example = qa_data[i]
    question = example["question"]
    ground_truth_ids = ast.literal_eval(example["relevant_passage_ids"])
    ground_truth_ids = [int(x) for x in ground_truth_ids]

    top_k_indices = retrieve_top_k(question, corpus_embeddings, k=k)
    retrieved_ids = [corpus_ids[idx] for idx in top_k_indices]

    recall, precision, f1 = compute_ir_metrics(retrieved_ids, ground_truth_ids)

    recalls.append(recall)
    precisions.append(precision)
    f1s.append(f1)

avg_recall = np.mean(recalls)
avg_precision = np.mean(precisions)
avg_f1 = np.mean(f1s)
print(f"Average Recall@{k}: {avg_recall:.3f}")
print(f"Average Precision@{k}: {avg_precision:.3f}")
print(f"Average F1@{k}: {avg_f1:.3f}")

Average Recall@5: 0.218
Average Precision@5: 0.440
Average F1@5: 0.284


In [33]:
# ==================================================
# DO NOT MODIFY
# ==================================================

PASS_THRESHOLD_RECALL = 0.18
PASS_THRESHOLD_PRECISION = 0.40
PASS_THRESHOLD_F1 = 0.24

if avg_recall < PASS_THRESHOLD_RECALL:
    raise ValueError(
        f"FAILED: Average Recall@5 too low ({avg_recall:.3f}). "
        "Check your cosine similarity and top-k retrieval implementation."
    )

if avg_precision < PASS_THRESHOLD_PRECISION:
    raise ValueError(
        f"FAILED: Average Precision@5 too low ({avg_precision:.3f}). "
        "Check your cosine similarity and top-k retrieval implementation."
    )

if avg_f1 < PASS_THRESHOLD_F1:
    raise ValueError(
        f"FAILED: Average F1@5 too low ({avg_f1:.3f}). "
        "Check your cosine similarity and top-k retrieval implementation."
    )

print("Part 2 passed")

Part 2 passed


## Section 3: Retrieval-Augmented Generation (15 points)

In this section, you will combine your retriever with the Llama model to implement a full RAG pipeline. You will then evaluate whether retrieval improves answer quality compared to Section 1.

### Task:
Use the same 10 examples from the `rag-mini-bioasq` dataset,
- Retrieve the top 5 passages using your dense retriever.
- Construct the following instruction prompt:
> You are a helpful chatbot.
> Use ONLY the following context to answer the question.
> Do not make up any new information.
>
> Context: [context]
>
> Question: [question]
>
> Answer:

- Generate answers using `Llama-3.2-1B-Instruct` with the instruction prompt.
- Compute BLEU score against ground-truth answers.


In [37]:
def build_rag_prompt(question, retrieved_passages):
    # TODO:
    # - Write a context-aware instruction prompt as a formatted string.
    # - Insert the variable `question`.
    # - Insert the retrieved passages from `retrieved_passages` (a list of strings, separated by newlines) as the context.
    # - Make sure the prompt ends with "Answer:"

    context = "\n".join(retrieved_passages)
    prompt = (
        "You are a helpful chatbot.Use ONLY the following context to answer the question.\n"
        "Do not make up any new information.\n\n"
        f"Context: {context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )
    return prompt

In [38]:
k = 5
rag_predictions = []
rag_references = []

for i in range(10):
    example = qa_data[i]
    question = example["question"]
    reference_answer = example["answer"]

    # TODO: 1. Retrieve top-k passages
    top_k_indices = retrieve_top_k(question, corpus_embeddings, k=k)
    retrieved_passages = [corpus_texts[idx] for idx in top_k_indices]
    # TODO: 2. Build context-aware prompt
    prompt = build_rag_prompt(question, retrieved_passages)
    # TODO: 3. Generate model answer
    answer = generate_answer(prompt, model, tokenizer)
    # TODO: 4. Store prediction and reference
    rag_predictions.append(answer)
    rag_references.append(reference_answer)
rag_bleu = compute_bleu(rag_predictions, rag_references)
print("Average BLEU (with retrieval):", rag_bleu)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end gene

Average BLEU (with retrieval): 0.10264736905421706


In [39]:
# ==================================================
# DO NOT MODIFY
# ==================================================
from evaluate import load
bleu = load("bleu")

if len(rag_predictions) != 10:
    raise ValueError("FAILED: You must generate 10 predictions.")

rag_bleu = bleu.compute(predictions=rag_predictions, references=rag_references)
print(f"Average BLEU for RAG answers: {rag_bleu['bleu']:.4f}")

# Optional: minimal threshold for grading
if rag_bleu["bleu"] < 0.09:
    raise ValueError("FAILED: BLEU score too low. Check your retrieval, prompt, or answer generation.")
print("Part 3 passed")

Average BLEU for RAG answers: 0.1026
Part 3 passed
